# Phase 4.7 — Re-ID Performance Benchmark (GPU)

**Goal:** Measure the actual time required for OSNet to process the complete 353-crop TRACE dataset on Google Colab Tesla T4 GPU.

This benchmark uses:
- OSNet x1_0 with pretrained weights
- Batch size 32 (same as CPU benchmark)
- All 353 real TRACE crops
- CUDA synchronization for accurate timing

**Runtime requirement:** GPU — `Runtime → Change runtime type → GPU`

## Step 1 — Verify GPU Environment

In [ ]:
import torch

print("=" * 50)
print("ENVIRONMENT CHECK")
print("=" * 50)
print(f"PyTorch version : {torch.__version__}")
print(f"CUDA available  : {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU name        : {torch.cuda.get_device_name(0)}")
    print(f"CUDA version    : {torch.version.cuda}")
    mem_total = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"GPU memory      : {mem_total:.1f} GB")
    device = torch.device('cuda')
else:
    print("ERROR: CUDA not available — GPU is required for this benchmark.")
    print("Go to Runtime → Change runtime type → GPU and re-run.")
    raise SystemExit("GPU required")

print(f"\nUsing device    : {device}")
print("=" * 50)

## Step 2 — Install Dependencies

In [ ]:
# Install torchreid from the official KaiyangZhou repository
!pip install -q git+https://github.com/KaiyangZhou/deep-person-reid.git

# Also ensure Pillow and numpy are present
!pip install -q Pillow numpy

# -------------------------------------------------------
# IMPORTANT: After this cell finishes, go to:
#   Runtime → Restart session
# Then run ALL cells again from Step 1.
# -------------------------------------------------------
print("Installation complete. Please restart the runtime now (Runtime → Restart session).")
print("After restart, run all cells from Step 1.")

## Step 3 — Import Libraries

In [ ]:
import os
import json
import math
import time
import statistics
from datetime import datetime
from pathlib import Path
from typing import NamedTuple

import numpy as np
from PIL import Image

import torch
import torch.nn.functional as F
import torchvision.transforms as T

import torchreid

print("All imports successful.")
torchreid_version = getattr(torchreid, '__version__', 'installed from source (no __version__)')
print(f"torchreid version : {torchreid_version}")

## Step 4 — Mount Google Drive and Locate TRACE Dataset

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# -------------------------------------------------------
# UPDATE THIS PATH to where TRACE lives in your Drive
# Example: '/content/drive/MyDrive/Trace'
# -------------------------------------------------------
TRACE_ROOT = '/content/drive/MyDrive/Trace'  # <-- adjust if needed

CROPS_DIR      = os.path.join(TRACE_ROOT, 'dataset', 'crops_phase3_final')
METADATA_PATH  = os.path.join(TRACE_ROOT, 'dataset', 'crops_metadata_phase3_final.json')

# Verify paths exist
assert os.path.isdir(CROPS_DIR), (
    f"Crops directory not found: {CROPS_DIR}\n"
    "Please update TRACE_ROOT above."
)
assert os.path.isfile(METADATA_PATH), (
    f"Metadata file not found: {METADATA_PATH}\n"
    "Please update TRACE_ROOT above."
)

print(f"Crops directory  : {CROPS_DIR}")
print(f"Metadata file    : {METADATA_PATH}")

# Count available crops
crop_files = sorted([
    f for f in os.listdir(CROPS_DIR)
    if f.lower().endswith('.jpg')
])
print(f"Crops found      : {len(crop_files)}")

## Step 5 — Load Metadata

In [ ]:
with open(METADATA_PATH, 'r') as f:
    metadata = json.load(f)

print(f"Metadata entries : {len(metadata)}")

# Collect crop file paths
crop_paths = []
for rec in metadata:
    crop_filename = os.path.basename(rec['crop_path'])
    crop_path = os.path.join(CROPS_DIR, crop_filename)
    if not os.path.isfile(crop_path):
        raise FileNotFoundError(f"Crop file not found: {crop_path}")
    crop_paths.append(crop_path)

print(f"Crop paths loaded: {len(crop_paths)}")

## Step 6 — Define Preprocessing (Same as CPU Benchmark)

In [ ]:
# OSNet expected input resolution (H x W)
INPUT_H = 256
INPUT_W = 128

# Standard ImageNet normalization used during OSNet pretraining
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

transform = T.Compose([
    T.Resize((INPUT_H, INPUT_W)),          # resize to 256x128
    T.ToTensor(),                           # HWC uint8 [0,255] → CHW float [0.0,1.0]
    T.Normalize(mean=IMAGENET_MEAN,
                std=IMAGENET_STD),          # per-channel normalization
])

def preprocess_image(image_path):
    """Load and preprocess a single crop image."""
    img = Image.open(image_path).convert('RGB')
    tensor = transform(img)
    return tensor

print("Preprocessing pipeline defined.")

## Step 7 — Load OSNet Model

In [ ]:
MODEL_NAME = 'osnet_x1_0'
BATCH_SIZE = 32
NUM_RUNS = 3

print(f"\nLoading {MODEL_NAME} with pretrained weights...")
t_load_start = time.perf_counter()

model = torchreid.models.build_model(
    name=MODEL_NAME,
    num_classes=1000,
    pretrained=True,
)
model.to(device)
model.eval()

model_load_time = time.perf_counter() - t_load_start
print(f"Model loaded in {model_load_time:.3f}s on {device}")

## Step 8 — Determine Embedding Dimension

In [ ]:
print("Probing model output dimension...")
with torch.no_grad():
    dummy = torch.zeros(1, 3, INPUT_H, INPUT_W, device=device)
    probe = model(dummy)
    embedding_dim = probe.shape[1]
print(f"Model output dim: {embedding_dim}")

## Step 9 — Warm-up (Not Counted in Timing)

In [ ]:
print("Performing warm-up...")
t_warmup_start = time.perf_counter()

# Process a small batch for warm-up
warmup_size = min(4, len(crop_paths))
warmup_tensors = []
for i in range(warmup_size):
    warmup_tensors.append(preprocess_image(crop_paths[i]))

if warmup_tensors:
    warmup_batch = torch.stack(warmup_tensors).to(device)
    with torch.no_grad():
        _ = model(warmup_batch)

torch.cuda.synchronize()
warmup_time = time.perf_counter() - t_warmup_start
print(f"Warm-up completed in {warmup_time:.3f}s")

## Step 10 — Benchmark Runs

In [ ]:
total_crops = len(crop_paths)
n_batches = math.ceil(total_crops / BATCH_SIZE)

print(f"\n{'=' * 60}")
print(f"GPU BENCHMARK CONFIGURATION")
print(f"{'=' * 60}")
print(f"GPU              : {torch.cuda.get_device_name(0)}")
print(f"Crops            : {total_crops}")
print(f"Batch size       : {BATCH_SIZE}
print(f"Number of runs   : {NUM_RUNS}")
print(f"Model            : {MODEL_NAME}")
print(f"Embedding dim    : {embedding_dim}")
print(f"{'=' * 60}\n")

runs = []

for run_idx in range(1, NUM_RUNS + 1):
    print(f"{'=' * 60}")
    print(f"Run {run_idx}/{NUM_RUNS}")
    print(f"{'=' * 60}")
    
    # Preprocessing timing
    t_preprocess_start = time.perf_counter()
    all_tensors = []
    for crop_path in crop_paths:
        tensor = preprocess_image(crop_path)
        all_tensors.append(tensor)
    preprocessing_time = time.perf_counter() - t_preprocess_start
    
    # Inference timing with CUDA synchronization
    torch.cuda.synchronize()
    t_inference_start = time.perf_counter()
    
    for batch_idx in range(n_batches):
        batch_start = batch_idx * BATCH_SIZE
        batch_end = min(batch_start + BATCH_SIZE, total_crops)
        batch_tensors = all_tensors[batch_start:batch_end]
        
        batch_tensor = torch.stack(batch_tensors).to(device)
        with torch.no_grad():
            _ = model(batch_tensor)
    
    torch.cuda.synchronize()
    inference_time = time.perf_counter() - t_inference_start
    
    # Total time for this run
    total_time = preprocessing_time + inference_time
    
    # Calculate metrics
    crops_per_second = total_crops / total_time
    ms_per_crop = 1000 * total_time / total_crops
    
    run_result = {
        'run_number': run_idx,
        'total_time': total_time,
        'model_load_time': model_load_time,
        'warmup_time': warmup_time,
        'inference_time': inference_time,
        'preprocessing_time': preprocessing_time,
        'crops_processed': total_crops,
        'crops_per_second': crops_per_second,
        'ms_per_crop': ms_per_crop,
    }
    runs.append(run_result)
    
    print(f"Total time        : {total_time:.3f}s")
    print(f"Preprocessing     : {preprocessing_time:.3f}s")
    print(f"Inference         : {inference_time:.3f}s")
    print(f"Crops/sec         : {crops_per_second:.1f}")
    print(f"ms/crop           : {ms_per_crop:.2f}")
    print()

## Step 11 — Calculate Summary Statistics

In [ ]:
total_times = [r['total_time'] for r in runs]
inference_times = [r['inference_time'] for r in runs]
crops_per_sec_values = [r['crops_per_second'] for r in runs]
ms_per_crop_values = [r['ms_per_crop'] for r in runs]

summary = {
    'mean_total_time': statistics.mean(total_times),
    'median_total_time': statistics.median(total_times),
    'min_total_time': min(total_times),
    'max_total_time': max(total_times),
    'mean_inference_time': statistics.mean(inference_times),
    'mean_crops_per_second': statistics.mean(crops_per_sec_values),
    'mean_ms_per_crop': statistics.mean(ms_per_crop_values),
    'total_runs': NUM_RUNS,
}

print("\n" + "=" * 60)
print("GPU BENCHMARK SUMMARY")
print("=" * 60)
print(f"Environment          : {device}")
print(f"GPU                  : {torch.cuda.get_device_name(0)}")
print(f"Crops                : {total_crops}")
print(f"Batch size           : {BATCH_SIZE}")
print(f"Model                : {MODEL_NAME}")
print(f"Embedding dim        : {embedding_dim}")
print(f"Model load time      : {model_load_time:.3f}s")
print(f"Warm-up time         : {warmup_time:.3f}s")
print()
print(f"Runs                 : {summary['total_runs']}")
print(f"Mean total time      : {summary['mean_total_time']:.3f}s")
print(f"Median total time    : {summary['median_total_time']:.3f}s")
print(f"Min total time       : {summary['min_total_time']:.3f}s")
print(f"Max total time       : {summary['max_total_time']:.3f}s")
print(f"Mean inference time  : {summary['mean_inference_time']:.3f}s")
print(f"Mean crops/sec       : {summary['mean_crops_per_second']:.1f}")
print(f"Mean ms/crop         : {summary['mean_ms_per_crop']:.2f}")
print("=" * 60)

## Step 12 — Save Results

In [ ]:
# Prepare results for saving
benchmark_result = {
    'timestamp': datetime.now().isoformat(),
    'environment': {
        'pytorch_version': torch.__version__,
        'cuda_available': torch.cuda.is_available(),
        'device': str(device),
        'gpu_name': torch.cuda.get_device_name(0),
        'cuda_version': torch.version.cuda,
        'gpu_memory_gb': round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2),
        'model': MODEL_NAME,
        'embedding_dim': embedding_dim,
        'crop_count': total_crops,
        'batch_size': BATCH_SIZE,
    },
    'model_load_time': model_load_time,
    'warmup_time': warmup_time,
    'runs': runs,
    'summary': summary,
}

# Save to Google Drive
output_dir = os.path.join(TRACE_ROOT, 'dataset', 'reid_benchmark')
os.makedirs(output_dir, exist_ok=True)
output_path = os.path.join(output_dir, 'performance_benchmark_gpu.json')

with open(output_path, 'w') as f:
    json.dump(benchmark_result, f, indent=2)

print(f"\nGPU benchmark results saved to: {output_path}")

## Step 13 — Final Report

In [ ]:
print("\n" + "=" * 60)
print("PHASE 4.7 GPU BENCHMARK — FINAL REPORT")
print("=" * 60)
print(f"Model               : {MODEL_NAME}")
print(f"Pretrained          : Yes")
print(f"Device              : {device}")
print(f"GPU                 : {torch.cuda.get_device_name(0)}")
print(f"Crops               : {total_crops}")
print(f"Batch size          : {BATCH_SIZE}")
print(f"Embedding dimension : {embedding_dim}")
print()
print("TIMING RESULTS")
print("-" * 60)
for i, run in enumerate(runs, 1):
    print(f"Run {i}: {run['total_time']:.3f}s ({run['crops_per_second']:.1f} crops/sec, {run['ms_per_crop']:.2f} ms/crop)")
print()
print(f"Mean total time     : {summary['mean_total_time']:.3f}s")
print(f"Mean inference time : {summary['mean_inference_time']:.3f}s")
print(f"Mean crops/sec      : {summary['mean_crops_per_second']:.1f}")
print(f"Mean ms/crop        : {summary['mean_ms_per_crop']:.2f}")
print("=" * 60)
print()
print("Phase 4.7 GPU benchmark is COMPLETE.")
print("Results saved to Google Drive.")